In [1]:
# Cell 1 — Imports & setup
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import io
import zipfile
import pandas as pd

EIA_URL = "https://www.eia.gov/electricity/data/eia923/"
YEAR_PATTERN = re.compile(r"(20[0-2][0-9])")

# Pattern to find the main EIA-923 Excel file inside each ZIP
def excel_pattern(year):
    return re.compile(
        rf"EIA923[\s_]+SCHEDULES?[\s_]+2[\s_]3[\s_]4[\s_]5.*{year}.*\.xlsx?",
        re.IGNORECASE,
    )

print("Setup complete.")

/Users/nicholasabad/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Setup complete.


In [2]:
# Cell 2 — Discover ZIP URLs
response = requests.get(EIA_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")
links = soup.find_all("a", href=True)

zip_files = []
for link in links:
    href = link["href"]
    text = link.get_text(strip=True)
    if ".zip" in href.lower():
        year_match = YEAR_PATTERN.search(text + " " + href)
        if year_match:
            year = int(year_match.group(1))
            if 2009 <= year:
                full_url = urljoin(EIA_URL, href)
                zip_files.append({"year": year, "url": full_url})

# Deduplicate by year (keep first occurrence = latest on page)
seen_years = set()
zip_by_year = {}
for entry in sorted(zip_files, key=lambda x: x["year"], reverse=True):
    if entry["year"] not in seen_years:
        seen_years.add(entry["year"])
        zip_by_year[entry["year"]] = entry["url"]

print(f"Discovered {len(zip_by_year)} years: {sorted(zip_by_year.keys())}")

Discovered 17 years: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [3]:
# Cell 3 — Download & extract Plant Frame sheets
all_frames = []

def extract_plant_frame(file_handle, year):
    """Try multiple header rows and column name variants to find Plant Id/Name."""
    for header_row in [3, 4, 5]:
        try:
            df = pd.read_excel(
                file_handle,
                sheet_name="Page 6 Plant Frame",
                header=header_row,
            )
            # Normalize column names: strip whitespace
            df.columns = [str(c).strip() for c in df.columns]

            # Find Plant Id and Plant Name columns (may vary by year)
            id_col = None
            name_col = None
            for col in df.columns:
                if col.lower().replace(" ", "") in ("plantid", "plantcode"):
                    id_col = col
                elif col.lower().replace(" ", "") in ("plantname",):
                    name_col = col

            if id_col and name_col:
                result = df[[id_col, name_col]].copy()
                result.columns = ["Plant Id", "Plant Name"]
                result = result.dropna(subset=["Plant Id"]).drop_duplicates(subset=["Plant Id"])
                return result

            # Reset file position for next attempt
            file_handle.seek(0)
        except Exception:
            file_handle.seek(0)
            continue
    return None

for year in sorted(zip_by_year.keys()):
    url = zip_by_year[year]
    try:
        print(f"Downloading {year}...", end=" ")
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            # Find the main EIA-923 Excel file
            pat = excel_pattern(year)
            excel_files = [n for n in zf.namelist() if pat.search(n)]

            if not excel_files:
                # Fallback: try any xlsx/xls file with the year
                fallback_pat = re.compile(rf".*{year}.*\.xlsx?", re.IGNORECASE)
                excel_files = [n for n in zf.namelist() if fallback_pat.search(n)]

            if not excel_files:
                print(f"no matching Excel file found. Files: {zf.namelist()[:5]}")
                continue

            excel_name = excel_files[0]
            with zf.open(excel_name) as f:
                data = io.BytesIO(f.read())

            df = extract_plant_frame(data, year)
            if df is not None:
                df["year"] = year
                all_frames.append(df)
                print(f"OK — {len(df)} plants")
            else:
                print(f"could not find Plant Id/Plant Name columns")

    except Exception as e:
        print(f"error: {e}")

plant_df = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal rows (all years combined): {len(plant_df)}")

could not find Plant Id/Plant Name columns

could not find Plant Id/Plant Name columns

could not find Plant Id/Plant Name columns

OK — 6507 plants

could not find Plant Id/Plant Name columns

...

OK — 11296 plants

OK — 11881 plants

OK — 12560 plants

OK — 13418 plants

OK — 14033 plants

Total rows (all years combined): 131002


In [4]:
# Cell 4 — Build final lookup (keep latest year's name per plant)
plant_df = plant_df.sort_values("year", ascending=True)
lookup = plant_df.drop_duplicates(subset=["Plant Id"], keep="last")[["Plant Id", "Plant Name"]].copy()
lookup = lookup.rename(columns={"Plant Id": "plant_code", "Plant Name": "plant_name"})
lookup["plant_code"] = lookup["plant_code"].astype(int).astype(str)
lookup = lookup.sort_values("plant_code", key=lambda s: s.astype(int)).reset_index(drop=True)

print(f"Unique plant codes: {len(lookup)}")
print(f"\nSample rows:")
print(lookup.head(10))

Unique plant codes: 15407

Sample rows:
  plant_code          plant_name
0          1          Sand Point
1          2        Bankhead Dam
2          3               Barry
3          4  Walter Bouldin Dam
4          7             Gadsden
5          8              Gorgas
6          9              Copper
7         10       Greene County
8         11   H Neely Henry Dam
9         12            Holt Dam


In [5]:
# Cell 5 — Save
import os

output_path = os.path.join(os.path.dirname(os.getcwd()), "data", "crosswalks", "eia_plant_lookup.csv")
lookup.to_csv(output_path, index=False)
print(f"Saved {len(lookup)} rows to {output_path}")

Saved 15407 rows to /Users/nicholasabad/Desktop/workspace/consulting-christine/data/plant-data/data/crosswalks/eia_plant_lookup.csv
